In [26]:
from google.colab import drive
drive.mount("/content/drive")

!ls -lh /content/drive/MyDrive/retinopatia

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 507M
-rw------- 1 root root 507M Nov  7 17:43 aptos_384.zip
-rw------- 1 root root 253K Nov  7 20:37 particion.csv


In [27]:
!cp /content/drive/MyDrive/retinopatia/aptos_384.zip /content/
!unzip -q -o /content/aptos_384.zip -d /content/data
!ls /content/data/aptos_384
!ls /content/data/aptos_384/imagenes | wc -l

aptos_limpio.csv  imagenes  reportes
3583


In [28]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

SEMILLA = 42
NOMBRES = {0: "Sin RD", 1: "Leve", 2: "Moderada", 3: "Severa", 4: "Proliferativa"}
ORDEN = list(NOMBRES.values())

DATOS = Path("/content/data/aptos_384")
IMAGENES = DATOS / "imagenes"
DRIVE = Path("/content/drive/MyDrive/retinopatia")

df = pd.read_csv(DATOS / "aptos_limpio.csv")
df["path"] = df["id_code"].map(lambda x: IMAGENES / f"{x}.png")

assert df["path"].map(Path.exists).all(), "Faltan imágenes"
print(df.shape)

(3583, 9)


In [29]:
mezclados = df.groupby("grupo")["diagnosis"].nunique()
mezclados = mezclados[mezclados > 1].index
print("Grupos con etiquetas mezcladas:", list(mezclados))

mascara = df["grupo"].isin(mezclados)
df.loc[mascara, "grupo"] = df["grupo"].max() + 1 + np.arange(mascara.sum())

assert df.groupby("grupo")["diagnosis"].nunique().max() == 1
print("Ahora cada grupo tiene una sola etiqueta")

Grupos con etiquetas mezcladas: [24]
Ahora cada grupo tiene una sola etiqueta


In [30]:
grupos = df.groupby("grupo")["diagnosis"].first().reset_index()

g_train, g_temp = train_test_split(
    grupos, test_size=0.30, stratify=grupos["diagnosis"], random_state=SEMILLA
)
g_val, g_test = train_test_split(
    g_temp, test_size=0.50, stratify=g_temp["diagnosis"], random_state=SEMILLA
)

asignacion = {}
for nombre, g in [("train", g_train), ("val", g_val), ("test", g_test)]:
    asignacion.update(dict.fromkeys(g["grupo"], nombre))

df["particion"] = df["grupo"].map(asignacion)
print(df["particion"].value_counts())

particion
train    2507
val       542
test      534
Name: count, dtype: int64


In [31]:
df["fold"] = -1
idx_train = df.index[df["particion"] == "train"]

sgkf = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEMILLA)
divisiones = sgkf.split(
    idx_train,
    df.loc[idx_train, "diagnosis"],
    df.loc[idx_train, "grupo"],
)

for k, (_, pos_val) in enumerate(divisiones):
    df.loc[idx_train[pos_val], "fold"] = k

print(df.loc[idx_train, "fold"].value_counts())

fold
0    1254
1    1253
Name: count, dtype: int64


In [32]:
tabla = pd.crosstab(df["clase"], df["particion"]).reindex(ORDEN)[["train", "val", "test"]]
tabla.loc["Total"] = tabla.sum()
print("Imágenes por clase y partición:")
print(tabla)

print("\nPorcentaje de cada clase dentro de su partición:")
print((tabla / tabla.loc["Total"] * 100).round(1))

print("\nTamaño de cada partición (%):")
print((df["particion"].value_counts(normalize=True) * 100).round(1))

fuga = df.groupby("grupo")["particion"].nunique().gt(1).sum()
fuga_folds = df[df["particion"] == "train"].groupby("grupo")["fold"].nunique().gt(1).sum()
print("\nGrupos repartidos entre particiones:", fuga)
print("Grupos repartidos entre folds:", fuga_folds)
assert fuga == 0 and fuga_folds == 0

print("\nClases por fold:")
print(pd.crosstab(df.loc[idx_train, "clase"], df.loc[idx_train, "fold"]).reindex(ORDEN))

print("\nImágenes de 2588x1958 por partición:")
print(df[df["tamano"] == "2588x1958"]["particion"].value_counts())

Imágenes por clase y partición:
particion      train  val  test
clase                          
Sin RD          1260  273   268
Leve             252   54    54
Moderada         683  148   144
Severa           120   27    25
Proliferativa    192   40    43
Total           2507  542   534

Porcentaje de cada clase dentro de su partición:
particion      train    val   test
clase                             
Sin RD          50.3   50.4   50.2
Leve            10.1   10.0   10.1
Moderada        27.2   27.3   27.0
Severa           4.8    5.0    4.7
Proliferativa    7.7    7.4    8.1
Total          100.0  100.0  100.0

Tamaño de cada partición (%):
particion
train    70.0
val      15.1
test     14.9
Name: proportion, dtype: float64

Grupos repartidos entre particiones: 0
Grupos repartidos entre folds: 0

Clases por fold:
fold             0    1
clase                  
Sin RD         637  623
Leve           119  133
Moderada       332  351
Severa          68   52
Proliferativa   98   94

Imágen

In [33]:
df.drop(columns="path").to_csv(DRIVE / "particion.csv", index=False)
print("Guardado en", DRIVE / "particion.csv")

Guardado en /content/drive/MyDrive/retinopatia/particion.csv


In [34]:
!ls -lh /content/drive/MyDrive/retinopatia
!echo "--- /content ---"
!ls /content
!echo "--- /content/data ---"
!ls /content/data
!echo "--- búsqueda ---"
!find /content -maxdepth 4 -name "aptos_limpio.csv" -not -path "/content/drive/*"

total 507M
-rw------- 1 root root 507M Nov  7 17:43 aptos_384.zip
-rw------- 1 root root 253K Nov  7 20:48 particion.csv
--- /content ---
aptos_384.zip  data  drive  sample_data
--- /content/data ---
aptos_384
--- búsqueda ---
/content/data/aptos_384/aptos_limpio.csv
